In [1]:
import ccxt
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import datetime
bybit = ccxt.bybit()
from freqtrade_client import FtRestClient
from datetime import datetime, date, timedelta, timezone, time
import pandas as pd
import os
server_url = 'http://127.0.0.1:8080'
username = ''
password = ""
client = FtRestClient(server_url, username, password)

In [21]:
trades = pd.DataFrame(client.trades().get('trades'))

starting_balance = client.daily(1).get('data')[0].get('starting_balance')
today_loss = trades[
    (trades.close_profit_abs < 0) & 
    (trades.close_date > date.today().strftime('%Y-%m-%d'))
].close_profit_abs.sum().item() / starting_balance

week_day = date.weekday(date.today())
open_date = (date.today() - timedelta(days=week_day))
starting_balance = client.weekly(1).get('data')[0].get('starting_balance')
this_week_loss = trades[
    (trades.close_profit_abs < 0) & 
    (trades.close_date > open_date.strftime('%Y-%m-%d'))
].close_profit_abs.sum().item() / starting_balance

In [ ]:
today_loss

In [ ]:
today_loss

In [ ]:
week_day = date.weekday(date.today())
open_date = (date.today() - timedelta(days=week_day))
open_date

In [ ]:
all_trades = client.trades().get('trades') + client.status()
dataframe = pd.DataFrame(all_trades)
dataframe.is_open.any()

In [3]:
def return_order_book(symbol='BTC/USDT', n=200):
    bybit_ob = bybit.fetchOrderBook(symbol, n)
    binance_ob = bybit.fetchOrderBook(symbol, n)
    kucoin_ob = bybit.fetchOrderBook(symbol, n)
    bid_values = {
        'price': np.hstack((np.array(bybit_ob['bids'])[:,0], np.array(binance_ob['bids'])[:,0], np.array(kucoin_ob['bids'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['bids'])[:,1], np.array(binance_ob['bids'])[:,1], np.array(kucoin_ob['bids'])[:,1])),
        'side':'bid'
    }
    ask_values = {
        'price': np.hstack((np.array(bybit_ob['asks'])[:,0], np.array(binance_ob['asks'])[:,0], np.array(kucoin_ob['asks'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['asks'])[:,1], np.array(binance_ob['asks'])[:,1], np.array(kucoin_ob['asks'])[:,1])),
        'side':'ask'
    }
    bid_dataframe = pd.DataFrame(bid_values)
    ask_dataframe = pd.DataFrame(ask_values)
    dataframe = pd.concat((bid_dataframe,ask_dataframe))
    dataframe = dataframe.groupby(['price','side']).sum().reset_index()
    # dataframe.groupby('side').sum()
    dataframe['now'] = datetime.datetime.now()
    return dataframe

In [3]:
def plot(dataframe, trades=[]):

    fig = go.Figure(data=[go.Candlestick(x=dataframe.date.values,
                    open=dataframe['open'],
                    high=dataframe['high'],
                    low=dataframe['low'],
                    close=dataframe['close'],
                    increasing_line_color= 'green', 
                    decreasing_line_color= 'red')])

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.upper_band.values,
        mode="lines", 
        marker=dict(size=7, color="green")
    )

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.y.values,
        mode="lines", 
        marker=dict(size=7, color="blue")
    )

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.lower_band.values,
        mode="lines", 
        marker=dict(size=7, color="red")
    )

    fig.add_scatter(
        x= dataframe[dataframe.extrema == 1].date.values, 
        y= dataframe[dataframe.extrema == 1].high.values,
        mode="markers", 
        marker=dict(size=7, color="purple")
    )

    fig.add_scatter(
        x= dataframe[dataframe.extrema == -1].date.values, 
        y= dataframe[dataframe.extrema == -1].low.values,
        mode="markers", 
        marker=dict(size=7, color="yellow")
    )

    if not trades.empty:
        fig.add_scatter(
            x= trades.open_date.values, 
            y= trades.open_rate.values,
            mode="markers", 
            marker=dict(size=15, color="green")
        )

        fig.add_scatter(
            x= trades.close_date.values, 
            y= trades.close_rate.values,
            mode="markers", 
            marker=dict(size=15, color="red")
        )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    fig.update_layout(autosize=True, height=500,xaxis_rangeslider_visible=False)
    fig.show()

In [4]:
def calculate_extrema(dataframe, kernel=6):
    dataframe["extrema"] = 0
    min_peaks = argrelextrema(dataframe["low"].values, np.less_equal, order=kernel)
    max_peaks = argrelextrema(dataframe["high"].values, np.greater_equal, order=kernel)
    for mp in min_peaks[0]:
        dataframe.at[mp, "extrema"] = -1
    for mp in max_peaks[0]:
        dataframe.at[mp, "extrema"] = 1
    dataframe['last_min_peak'] = dataframe.at[min_peaks[0][-1], "low"]
    dataframe['last_max_peak'] = dataframe.at[max_peaks[0][-1], "high"]
    dataframe['h_dist'] = np.where(dataframe.extrema == 1, (dataframe.high - dataframe.upper_band), 0)
    dataframe['l_dist'] = np.where(dataframe.extrema == -1, (dataframe.lower_band - dataframe.low), 0)
    dataframe['h_ratio'] = dataframe['h_dist'] / dataframe['band_dist']
    dataframe['l_ratio'] = dataframe['l_dist'] / dataframe['band_dist']
    dataframe['l_h_ratio'] = dataframe.at[max_peaks[0][-1], "h_ratio"]
    dataframe['l_l_ratio'] = dataframe.at[min_peaks[0][-1], "l_ratio"]
    dataframe['last_max'] = dataframe.at[max_peaks[0][-1], "close"]
    dataframe['last_min'] = dataframe.at[min_peaks[0][-1], "close"]
    return dataframe

In [5]:
def caculate_regression(dataframe, kernel=1440):
    dataframe_ = dataframe.copy()[-kernel:]
    x = dataframe_.index.values.reshape(-1, 1)
    y = dataframe_.close.values
    model = LinearRegression()
    model.fit(x, y)
    x = dataframe.index.values.reshape(-1, 1)
    dataframe['y'] = model.predict(x)
    dataframe['coef'] = float(model.coef_[0])
    dataframe['upper_band'] = dataframe['y'] + dataframe.high.std()
    dataframe['lower_band'] = dataframe['y'] - dataframe.low.std()
    dataframe['band_dist'] = dataframe['upper_band'] - dataframe['lower_band']
    return dataframe

In [ ]:
!docker-compose run --rm TradeStrategy download-data -c user_data/config.json --timeframe 1m

In [13]:
def return_dataframe_from_csv(pair, columns=[]):
    dataframe = pd.read_csv(f'df_{pair}.csv')
    dataframe['date'] = pd.to_datetime(dataframe['date'])
    if columns:
        dataframe = dataframe[columns]
    return dataframe

In [30]:
files = [file for file in os.listdir(".") if file.endswith('.csv')]
tickers = [file[3:-4] for file in files]

In [36]:
def plot_ticker(ticker):
# dataframe = pd.read_feather("../data/bybit/futures/BTC_USDT_USDT-1m-futures.feather")
    columns=['date','open','high','low','close','volume']
    dataframe = return_dataframe_from_csv(ticker)
    # open_time = '2024-11-19 12:42'
    # close_time = '2024-11-20 12:42'
    now = datetime.datetime.now()
    close_time = now.strftime("%Y-%m-%d %H:%M:%S")
    open_time = (now - datetime.timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
    # dataframe = dataframe[(dataframe.date > open_time) & (dataframe.date <= close_time)].reset_index()
    # dataframe = caculate_regression(dataframe, kernel=1440)
    # dataframe = calculate_extrema(dataframe, kernel=6)
    trades = pd.DataFrame(client.trades().get('trades') + client.status())
    trades = trades[(trades.open_date > open_time) & (trades.close_date <= close_time)].reset_index()
    last_candle = dataframe.iloc[-1].squeeze()
    print(float((last_candle['upper_band'] - last_candle['y']) / (last_candle['y'] - last_candle['lower_band'])))
    print("Coef:", dataframe.iloc[-1].squeeze()['coef'])
    print(ticker)
    plot(dataframe,trades=pd.DataFrame())

In [ ]:
plot_ticker(tickers[7])

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
dataframe = pd.read_csv("AVAX_8_df.csv")
# trades = pd.read_csv(f'trades.csv')

In [ ]:
print("Down: ", dataframe['down'].iloc[-240:].count())
print("Up: ", dataframe['up'].iloc[-240:].count())

In [ ]:
dataframe[["&s-up_or_down","down","up","enter_long", "enter_short", "do_predict"]].iloc[-12:]["&s-up_or_down"]

In [ ]:
import os

tickers = trades.pair.to_dict()
dfs = [f"{value[:-10]}_{key + 1}_df.csv" for  key, value in tickers.items()]
obs = [f"{value[:-10]}_{key + 1}_ob.csv" for  key, value in tickers.items()]
files = sorted(dfs + obs)
files.append(["analysis.ipynb", "trades.csv"])
for f in os.listdir('.'):
    if f not in files:
        os.unlink(f)

In [11]:
def caculate_coef(window):
    x = np.arange(len(window)).reshape(-1, 1)
    y = window
    model = LinearRegression()
    model.fit(x, y)
    x = dataframe.index.values.reshape(-1, 1)
    return model.coef_[0]

def calculate_coef_window(dataframe, window):
    dataframe['coef'] = dataframe.close.rolling(window=window).apply(caculate_coef)
    return dataframe


In [ ]:
fig = make_subplots(rows=3, cols=1)

dataframe = pd.read_csv("XRP_2_df.csv")
dataframe = dataframe[['date','open','high','low','close','volume']]

dataframe = calculate_coef_window(dataframe, window=12)

fig.add_trace(
    go.Scatter(
        x= dataframe.date.values, 
        y= dataframe.close.values,
        mode="lines", 
        marker=dict(size=7, color="green")
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x= dataframe.date.values, 
        y= dataframe.coef.values,
        mode="lines", 
        marker=dict(size=7, color="blue")
    ),
    row=3, col=1
)

fig.show()

In [7]:
import os
import re
import pandas as pd

In [95]:
def correlated_ticker(ticker):
    path = "../data/bybit/futures"
    all_files = os.listdir(path)
    files = [file for file in all_files if '_USDT_USDT-15m-futures' in file]
    tickers = {file[:-30]:pd.read_feather(path + "/" + file).close for file in files}
    dataframe = pd.DataFrame(tickers).drop(['BTC','ETH'], axis=1).ffill()
    corr_df = dataframe.corr()
    if ticker in corr_df.columns:
        corr_df = corr_df[(corr_df != 1)].dropna(how='all', axis=0).dropna(how='all', axis=1)
        return corr_df[ticker][corr_df[ticker] == corr_df[ticker].max()].to_dict()

In [105]:
correlated_ticker('AVAX')

{'XRP': 0.9562888734341671}

In [103]:
path = "../data/bybit/futures"
all_files = os.listdir(path)
files = [file for file in all_files if '_USDT_USDT-15m-futures' in file]
tickers = {file[:-30]:pd.read_feather(path + "/" + file).close for file in files}
dataframe = pd.DataFrame(tickers).drop(['BTC','ETH'], axis=1).ffill()

In [113]:
df_ = dataframe[['AVAX','XRP']]
df_

,AVAX,XRP
0,28.410,0.5533
1,28.450,0.5520
2,28.340,0.5513
3,28.375,0.5508
4,28.320,0.5499
...,...,...
3248,52.790,2.4177
3249,52.570,2.4179
3250,52.310,2.4188
3251,52.845,2.4415


In [127]:
((df_['AVAX'] - df_['AVAX'].mean()) / df_['AVAX'].std()).mean()

np.float64(5.591728876562731e-16)

In [126]:
((df_['XRP'] - df_['XRP'].mean()) / df_['XRP'].std()).mean()

np.float64(2.7958644382813655e-16)

In [116]:
print('XRP: ',df_['XRP'].std())
print('AVAX: ', df_['AVAX'].std())

XRP:  0.6871105669626805
AVAX:  7.42477176904112


In [117]:
df_['AVAX'].std() / df_['XRP'].std()

np.float64(10.805788945819527)